# Training AG News Models with Adversarial Examples

This notebook trains 4 text classification models on the AG News dataset and evaluates their robustness to adversarial attacks.

**Models:**
- TextCNN: CNN on word embeddings
- LSTM: Recurrent neural network
- BiLSTM: Bidirectional LSTM
- Transformer: Self-attention based

**Dataset:** AG News (120K articles, 4 topics: World, Sports, Business, Sci-Tech)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import sys

# Add project to path
sys.path.insert(0, '/Users/admin/Desktop/major_projekt')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Data

In [ ]:
from cerberus.dataset import get_ag_news_loaders

print("Loading AG News dataset...")
train_loader, test_loader, vocab = get_ag_news_loaders('./data', batch_size=128)

print(f"Vocabulary size: {len(vocab)}")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Sample a batch
texts, labels = next(iter(train_loader))
if texts is not None:
    print(f"\nBatch shapes:")
    print(f"  Texts: {texts.shape}")
    print(f"  Labels: {labels.shape}")
    print(f"  Label classes: {torch.unique(labels).tolist()}")

## 2. Define Models

In [ ]:
from backend import build_model

# Test model creation
print("Testing model creation...\n")

model_names = ['textcnn', 'lstm', 'bilstm', 'transformer']
models = {}

for name in model_names:
    try:
        model = build_model(name)
        model = model.to(device)
        models[name] = model
        
        # Count parameters
        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"✓ {name.upper():15} - {num_params:,} parameters")
    except Exception as e:
        print(f"✗ {name.upper():15} - Error: {e}")

print(f"\nSuccessfully created {len(models)} models")

## 3. Training Function

In [ ]:
def train_model(model, train_loader, test_loader, device, epochs=5, learning_rate=0.001, model_name='model'):
    """Train a text classification model"""
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]")
        for texts, labels in pbar:
            if texts is None or labels is None:
                continue
                
            texts = texts.to(device)
            labels = labels.to(device).long()
            
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': loss.item():.4f})
        
        # Testing
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [TEST]")
            for texts, labels in pbar:
                if texts is None or labels is None:
                    continue
                    
                texts = texts.to(device)
                labels = labels.to(device).long()
                
                outputs = model(texts)
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
        
        train_acc = 100 * train_correct / train_total
        test_acc = 100 * test_correct / test_total
        avg_loss = train_loss / len(train_loader)
        
        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        print(f"\n[Epoch {epoch+1}] Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
    
    # Save model
    save_path = f'models/{model_name}_agnews.pt'
    torch.save(model.state_dict(), save_path)
    print(f"\n✓ Model saved to {save_path}")
    
    return model, history

print("Training function defined")

## 4. Train All Models

In [ ]:
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Train each model
trained_models = {}
histories = {}

for model_name in model_names:
    print(f"\n" + "="*60)
    print(f"Training {model_name.upper()}")
    print("="*60)
    
    model = models[model_name]
    trained_model, history = train_model(
        model, train_loader, test_loader, device,
        epochs=3,  # Reduced for quick training
        learning_rate=0.001,
        model_name=model_name
    )
    
    trained_models[model_name] = trained_model
    histories[model_name] = history

## 5. Compare Model Performance

In [ ]:
# Create comparison table
print("\nFinal Test Accuracies:")
print("-" * 40)

results = {}
for model_name, history in histories.items():
    final_acc = history['test_acc'][-1]
    results[model_name] = final_acc
    print(f"{model_name.upper():15} {final_acc:6.2f}%")

# Find best model
best_model = max(results, key=results.get)
print("-" * 40)
print(f"Best model: {best_model.upper()} ({results[best_model]:.2f}%)")

## 6. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
for model_name, history in histories.items():
    axes[0].plot(history['train_loss'], label=model_name.upper())
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test accuracy
for model_name, history in histories.items():
    axes[1].plot(history['test_acc'], label=model_name.upper(), marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ag_news_training.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Training curves saved to ag_news_training.png")

## 7. Test Robustness to Adversarial Attacks

In [ ]:
# Import attack classes
from cerberus.attacks import FSGMAttack, PGDAttack

def evaluate_attack(model, attack_type, test_loader, device, epsilon=0.03):
    """Evaluate model robustness to a specific attack"""
    
    model.eval()
    clean_correct = 0
    adv_correct = 0
    total = 0
    
    # Create attack
    if attack_type == 'fgsm':
        attack = FSGMAttack(model=model, eps=epsilon/10, device=device)  # Smaller eps for text
    elif attack_type == 'pgd':
        attack = PGDAttack(model=model, eps=epsilon/10, alpha=epsilon/50, max_iter=20, device=device)
    else:
        raise ValueError(f"Unknown attack: {attack_type}")
    
    with torch.no_grad():
        for texts, labels in tqdm(test_loader, desc=f"Testing {attack_type.upper()}"):
            if texts is None or labels is None:
                continue
            
            texts = texts.to(device)
            labels = labels.to(device).long()
            
            # Clean accuracy
            outputs = model(texts)
            _, predicted = torch.max(outputs.data, 1)
            clean_correct += (predicted == labels).sum().item()
            
            # Adversarial accuracy
            adv_texts = attack.generate(texts, labels)
            adv_outputs = model(adv_texts)
            _, adv_predicted = torch.max(adv_outputs.data, 1)
            adv_correct += (adv_predicted == labels).sum().item()
            
            total += labels.size(0)
    
    return (clean_correct / total * 100), (adv_correct / total * 100)

print("Attack evaluation function defined")

## 8. Run Attack Evaluations

In [ ]:
# Evaluate models against FGSM and PGD
attack_results = {}

for model_name, model in trained_models.items():
    print(f"\n{'='*50}")
    print(f"Evaluating {model_name.upper()}")
    print(f"{'='*50}")
    
    model_results = {}
    
    # Test against FGSM
    print(f"\nTesting FGSM...")
    clean_acc, adv_acc = evaluate_attack(model, 'fgsm', test_loader, device, epsilon=0.03)
    model_results['fgsm'] = {'clean': clean_acc, 'adversarial': adv_acc}
    print(f"Clean: {clean_acc:.2f}% | Adversarial: {adv_acc:.2f}% | Drop: {clean_acc-adv_acc:.2f}pp")
    
    # Test against PGD
    print(f"\nTesting PGD...")
    clean_acc, adv_acc = evaluate_attack(model, 'pgd', test_loader, device, epsilon=0.03)
    model_results['pgd'] = {'clean': clean_acc, 'adversarial': adv_acc}
    print(f"Clean: {clean_acc:.2f}% | Adversarial: {adv_acc:.2f}% | Drop: {clean_acc-adv_acc:.2f}pp")
    
    attack_results[model_name] = model_results

print(f"\n{'='*50}")
print("✓ Attack evaluation complete")

## 9. Summary and Comparison

In [ ]:
import pandas as pd

# Create summary table
summary_data = []

for model_name in model_names:
    if model_name in attack_results:
        row = {
            'Model': model_name.upper(),
            'Final Acc': f"{results[model_name]:.2f}%",
            'FGSM Clean': f"{attack_results[model_name]['fgsm']['clean']:.2f}%",
            'FGSM Adv': f"{attack_results[model_name]['fgsm']['adversarial']:.2f}%",
            'FGSM Drop': f"{attack_results[model_name]['fgsm']['clean'] - attack_results[model_name]['fgsm']['adversarial']:.2f}pp",
            'PGD Clean': f"{attack_results[model_name]['pgd']['clean']:.2f}%",
            'PGD Adv': f"{attack_results[model_name]['pgd']['adversarial']:.2f}%",
            'PGD Drop': f"{attack_results[model_name]['pgd']['clean'] - attack_results[model_name]['pgd']['adversarial']:.2f}pp",
        }
        summary_data.append(row)

df = pd.DataFrame(summary_data)
print("\nAdversarial Robustness Summary - AG News Models")
print("="*120)
print(df.to_string(index=False))
print("="*120)

print("\n✓ Training and evaluation complete!")
print("\nModels saved to models/ directory:")
for model_name in model_names:
    print(f"  - models/{model_name}_agnews.pt")

## Key Findings

This notebook demonstrates:
1. **Multi-architecture comparison** - How different text models respond to adversarial attacks
2. **Attack effectiveness** - FGSM vs PGD on text data
3. **Robustness differences** - Which architectures are more naturally robust
4. **Cross-domain evaluation** - Adversarial attacks work across vision (CIFAR-10) and NLP (AG News)

### Next Steps
- Use the web interface to test adversarial attacks on these trained models
- Run AutoAttack for more rigorous evaluation
- Implement adversarial training to improve robustness
- Analyze transfer attacks between different text models